# S&P 500 Returns and U.S. Macroeconomic Indicators — Data Extraction

**Objective:** Build a clean monthly dataset (1990–2026) combining S&P 500 log returns with key macroeconomic variables (unemployment, Fed funds rate, industrial production, 10-year Treasury yield, and NBER recession indicator) to analyze how equity returns respond to labor market and monetary policy conditions across economic regimes.

**Output:** `sp500_macro_monthly_1990_2026.csv` / `.xlsx`

## 1. Extract S&P500 Data (Yahoo Finance)
Monthly closing prices for the S&P500 index (^GSPC) from 1990 - 2026, via 'yfinance'.

In [1]:
# Extract the SP500 data from Yahoo Finance using yfinance library
import yfinance as yf
import pandas as pd

sp500= yf.download("^GSPC", start="1990-01-01",end="2026-07-31",interval="1mo")

# The yfinance outcome is a multi-index dataframe, we need to drop the second level of the index to have a single level index.
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.droplevel(1)

sp500_close = sp500['Close'].rename("sp500_close")

print(sp500.columns)

[*********************100%***********************]  1 of 1 completed

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


## 2. Extract Macroeconomic Data (FRED)
Monthly macro data series retrieved from FRED API: unemployment rate (`UNRATE`), federal funds rate (`FEDFUNDS`), industrial production as a GDP proxy (`INDPRO`), 10-year Treasury yield (`GS10`), and the NBER recession indicator (`USREC`), and inflation ('CPIAUCSL')

In [2]:
# Get the FRED data using the freapi library and the FRED API key store in the .env file
from fredapi import Fred
from dotenv import load_dotenv
import os

load_dotenv()
fred = Fred(api_key=os.getenv('FRED_API_KEY'))

series_code = {
    "unemployment_rate": "UNRATE",
    "fed_rate": "FEDFUNDS",
    "indpro": "INDPRO",
    "gs10":"GS10",
    "usrec":"USREC",
    "cpi": "CPIAUCSL"
}

data = {}
for name, code in series_code.items():
    data[name] = fred.get_series(
        code, 
        observation_start="1990-01-01", 
        observation_end="2026-07-31"
        )

df_fred = pd.DataFrame(data)
df_fred 

,unemployment_rate,fed_rate,indpro,gs10,usrec,cpi
1990-01-01,5.4,8.23,61.7290,8.21,0.0,127.500
1990-02-01,5.3,8.24,62.2896,8.47,0.0,128.000
1990-03-01,5.2,8.28,62.5999,8.59,0.0,128.600
1990-04-01,5.4,8.26,62.4359,8.79,0.0,128.900
1990-05-01,5.4,8.18,62.6258,8.76,0.0,129.100
...,...,...,...,...,...,...
2026-03-01,4.3,3.64,101.6172,4.25,0.0,330.293
2026-04-01,4.3,3.64,102.4196,4.32,0.0,332.407
2026-05-01,4.3,3.63,102.5606,4.48,0.0,333.979
2026-06-01,4.2,3.63,102.6395,4.47,0.0,332.568


## 3. Align Date Indices
Both series are normalized to a common monthly timestamp format to ensure a clean merge

In [3]:
# Normalize the index of both dataframes to have a common format for merging
df_fred.index = df_fred.index.to_period('M').to_timestamp()
sp500.index = sp500.index.to_period('M').to_timestamp()

## 4. Merge Datasets 

In [4]:
# Merge the two dataframes on the index (date)
df_final = df_fred.join(sp500_close, how ='inner')
print(df_final.shape)
print(df_final.head())
print(df_final.isna().sum())


(439, 7)
            unemployment_rate  fed_rate   indpro  gs10  usrec    cpi  \
1990-01-01                5.4      8.23  61.7290  8.21    0.0  127.5   
1990-02-01                5.3      8.24  62.2896  8.47    0.0  128.0   
1990-03-01                5.2      8.28  62.5999  8.59    0.0  128.6   
1990-04-01                5.4      8.26  62.4359  8.79    0.0  128.9   
1990-05-01                5.4      8.18  62.6258  8.76    0.0  129.1   

            sp500_close  
1990-01-01   329.079987  
1990-02-01   331.890015  
1990-03-01   339.940002  
1990-04-01   330.799988  
1990-05-01   361.230011  
unemployment_rate    1
fed_rate             0
indpro               1
gs10                 0
usrec                0
cpi                  1
sp500_close          0
dtype: int64


## 5. Handle Missing Values
-unemployment rate and CPI had one missing value due to the 2025 U.S. government shutdown, filled via linear interpolation
-indpro is published with a reporting lag, the most recent data had not been released at the time of the extraction. 

In [5]:
# In 2025 there was a government shutdown generating a gap in the unemployment 
# and CPI data series (both published by the BLS). We use linear interpolation 
# to fill these single-month gaps.
df_final['unemployment_rate'] = df_final['unemployment_rate'].interpolate(method='linear')
df_final['cpi'] = df_final['cpi'].interpolate(method='linear')

print(df_final.isna().sum()) 


unemployment_rate    0
fed_rate             0
indpro               1
gs10                 0
usrec                0
cpi                  0
sp500_close          0
dtype: int64


In [6]:
df_final['inflation_yoy'] = df_final['cpi'].pct_change(periods=12)*100
print(df_final['inflation_yoy'].isna().sum())

12


In [7]:
df_final = df_final.drop(columns=['cpi']) 

## 6. Calculate S&P 500 Log Returns
The dependent variable is tranformed into log returns, the standard approach in financial econometrics for stabilizing variance and approximating stationary.

In [ ]:
import numpy as np
# Calculate sp500 returns and transform the dependent variable into log returns to make it stationary 
df_final['sp500_log_return'] = np.log(df_final['sp500_close'] / df_final['sp500_close'].shift(1))*100

#Additional transformations: log of INDPRO, this one behaves like a price/level index similar to sp500, kept alongside the level version so both can be compared in the stationarity tests (notebook 03), before deciding which one enters in the final model
df_final['log_indpro'] = np.log(df_final['indpro'])

In [ ]:
# Because we are calculating returns, the first row will be NaN, so we need to drop it.
df_final = df_final.dropna()

print(df_final.shape)

(426, 9)


## 7. Export Final Dataset
Save the cleaned dataset as both `.xlsx` and `.csv` for downstream analysis.

In [10]:
# Save the final dataframe as a xlsx file and CSV file too
df_final.to_excel("sp500_macro_monthly_1990_2026.xlsx")
df_final.to_csv("sp500_macro_monthly_1990_2026.csv")

In [11]:
df_final.head()

,unemployment_rate,fed_rate,indpro,gs10,usrec,sp500_close,inflation_yoy,sp500_log_return,log_indpro
1991-01-01,6.4,6.91,61.1355,8.09,1.0,343.929993,5.647059,4.067902,4.113093
1991-02-01,6.6,6.25,60.6838,7.85,1.0,367.070007,5.312500,6.511446,4.105677
1991-03-01,6.8,6.12,60.3346,8.11,1.0,375.220001,4.821151,2.195994,4.099906
1991-04-01,6.7,5.91,60.4938,8.04,0.0,375.339996,4.809930,0.031975,4.102541
1991-05-01,6.9,5.78,61.0633,8.07,0.0,389.829987,5.034857,3.787844,4.111911


In [12]:
df_final.tail()

,unemployment_rate,fed_rate,indpro,gs10,usrec,sp500_close,inflation_yoy,sp500_log_return,log_indpro
2026-02-01,4.4,3.64,101.9263,4.13,0.0,6878.879883,2.434004,-0.870613,4.624250
2026-03-01,4.3,3.64,101.6172,4.25,0.0,6528.520020,3.285958,-5.227556,4.621213
2026-04-01,4.3,3.64,102.4196,4.32,0.0,7209.009766,3.779246,9.915133,4.629078
2026-05-01,4.3,3.63,102.5606,4.48,0.0,7580.060059,4.166615,5.018952,4.630454
2026-06-01,4.2,3.63,102.6395,4.47,0.0,7499.359863,3.463531,-1.070346,4.631223


In [13]:
print(f"Exported Dataset: {df_final.shape[0]} monthly observations, {df_final.shape[1]} variables")
print(f"Range: {df_final.index.min()} to {df_final.index.max()}")

Exported Dataset: 426 monthly observations, 9 variables
Range: 1991-01-01 00:00:00 to 2026-06-01 00:00:00
